# 01 — Data Quality (10 chiều)

Notebook chuẩn M6a — gộp `docs/analysis/phase0_qc/` (phase0_data_check.py, phase0_full_qc.py,
phase0_qc_report.md) thành 1 notebook, gọi lại **10 hàm DQ thật** trong `datathon.quality`
(port ở M4a, commit `b3b85cf`) — KHÔNG viết lại thuật toán, chỉ tổ chức lại để chạy + hiển thị.

Nguồn dữ liệu: 14 bảng raw CSV qua `datathon.config.RAW_TABLES` (KHÔNG hardcode path).
Đọc file qua `datathon.ingestion.load_raw()` (parse ngày theo schema contract).

10 chiều (`RESTRUCTURE_AGENTS.md` — DATA QUALITY CHECK STANDARD):

| # | Hàm | Chiều | Severity khi sai |
|---|---|---|---|
| 1 | `inventory` | Snapshot/inventory | HARD_FAIL |
| 2 | `validate_schema` | Schema & format | QUARANTINE |
| 3 | `keys` | Grain & candidate key | HARD_FAIL |
| 4 | `check_ri` | Referential integrity | QUARANTINE |
| 5 | `completeness` | Completeness | WARN |
| 6 | `business_rules` | Business-rule validity | QUARANTINE |
| 7 | `time_coverage` | Time coverage | WARN |
| 8 | `dup_outlier` | Duplicate & outlier | QUARANTINE (dup) / FLAG (outlier) |
| 9 | `reconcile` | Cross-table reconciliation | HARD_FAIL |
| 10 | `write_dq_log` | Data-quality log | INFO (ghi log) |

**Lưu ý phạm vi**: notebook này chạy DQ ở tầng pandas trên **raw CSV** (đúng như M4a) — job
Iceberg thật (M2, `jobs/ingest_csv_to_iceberg.py`) gọi lại đúng các hàm này khi MERGE INTO
warehouse. Số liệu ở đây phải khớp `.process_status/M4a.md`.


In [1]:
import sys
import pandas as pd

from datathon import config, schema as schema_mod
from datathon.ingestion import load_raw, ingest_table
from datathon import quality as q

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

print("PROJECT_ROOT:", config.PROJECT_ROOT)
print("DATA_RAW    :", config.DATA_RAW)
print("14 bảng     :", schema_mod.ALL_TABLES)


PROJECT_ROOT: /Users/lhoanghai_/Documents/Study/Dự án datathon2026
DATA_RAW    : /Users/lhoanghai_/Documents/Study/Dự án datathon2026/data/raw
14 bảng     : ('customers', 'geography', 'products', 'promotions', 'orders', 'order_items', 'payments', 'shipments', 'returns', 'reviews', 'inventory', 'web_traffic', 'sales', 'sample_submission')


## 0. Load 14 bảng raw qua `config` + `load_raw`

In [2]:
tables = {}
for name in schema_mod.ALL_TABLES:
    try:
        tables[name] = load_raw(name)
    except FileNotFoundError as e:
        print("MISSING:", name, e)

shapes = pd.DataFrame(
    [{"table": t, "rows": len(df), "cols": len(df.columns)} for t, df in tables.items()]
).sort_values("table").reset_index(drop=True)
shapes


,table,rows,cols
0,customers,121930,7
1,geography,39948,4
2,inventory,60247,17
3,order_items,714669,7
4,orders,646945,8
5,payments,646945,4
6,products,2412,8
7,promotions,50,10
8,returns,39939,7
9,reviews,113551,7


In [3]:
BATCH_ID = q.new_batch_id()
print("batch_id cho toàn bộ notebook run:", BATCH_ID)
all_results = []


batch_id cho toàn bộ notebook run: 0be18a18d00d41868b73904ffa6bf8f5


## 1. Chiều 1 — `inventory` (Snapshot/inventory, HARD_FAIL)

14 bảng có mặt, số dòng > 0. Thiếu bảng hoặc bảng rỗng -> FAIL cứng, không đoán mò tiếp.


In [4]:
r1 = q.inventory(tables, BATCH_ID)
all_results += r1
pd.DataFrame([r.__dict__ for r in r1])


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,1,inventory,customers,row_count,121930,None,PASS,HARD_FAIL,"121,930 dòng, 7 cột.",0be18a18d00d41868b73904ffa6bf8f5
1,1,inventory,geography,row_count,39948,None,PASS,HARD_FAIL,"39,948 dòng, 4 cột.",0be18a18d00d41868b73904ffa6bf8f5
2,1,inventory,products,row_count,2412,None,PASS,HARD_FAIL,"2,412 dòng, 8 cột.",0be18a18d00d41868b73904ffa6bf8f5
3,1,inventory,promotions,row_count,50,None,PASS,HARD_FAIL,"50 dòng, 10 cột.",0be18a18d00d41868b73904ffa6bf8f5
4,1,inventory,orders,row_count,646945,None,PASS,HARD_FAIL,"646,945 dòng, 8 cột.",0be18a18d00d41868b73904ffa6bf8f5
5,1,inventory,order_items,row_count,714669,None,PASS,HARD_FAIL,"714,669 dòng, 7 cột.",0be18a18d00d41868b73904ffa6bf8f5
6,1,inventory,payments,row_count,646945,None,PASS,HARD_FAIL,"646,945 dòng, 4 cột.",0be18a18d00d41868b73904ffa6bf8f5
7,1,inventory,shipments,row_count,566067,None,PASS,HARD_FAIL,"566,067 dòng, 4 cột.",0be18a18d00d41868b73904ffa6bf8f5
8,1,inventory,returns,row_count,39939,None,PASS,HARD_FAIL,"39,939 dòng, 7 cột.",0be18a18d00d41868b73904ffa6bf8f5
9,1,inventory,reviews,row_count,113551,None,PASS,HARD_FAIL,"113,551 dòng, 7 cột.",0be18a18d00d41868b73904ffa6bf8f5


## 2. Chiều 2 — `validate_schema` (Schema & format, QUARANTINE)

dtype đúng + enum hợp lệ theo contract `schema.py`. Sai -> quarantine dòng vi phạm (không drop);
thiếu hẳn cột kỳ vọng -> FAIL (bảng không dùng được).


In [5]:
r2 = []
for t, df in tables.items():
    r2 += q.validate_schema(df, t, BATCH_ID)
all_results += r2
df2 = pd.DataFrame([r.__dict__ for r in r2])
print("PASS/WARN/FAIL:", df2["status"].value_counts().to_dict())
df2[df2["status"] != "PASS"]


PASS/WARN/FAIL: {'PASS': 14}


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id


## 3. Chiều 3 — `keys` (Grain & candidate key, HARD_FAIL)

`order_items` grain tự nhiên `(order_id, product_id)` KHÔNG unique (16 dòng hàng hợp lệ, khác
giá/số lượng) — đã biết trước, KHÔNG phải lỗi, nhưng bắt buộc phải có surrogate key unique để
MERGE. Kiểm tra 2 lần: (a) trên raw CSV thô (chưa có surrogate) — kỳ vọng FAIL đúng thiết kế;
(b) sau khi `ingest_table('order_items')` sinh surrogate — kỳ vọng PASS.


In [6]:
r3 = []
for t, df in tables.items():
    r3 += q.keys(df, t, BATCH_ID)
all_results += r3
df3 = pd.DataFrame([r.__dict__ for r in r3])
df3[df3["table"] == "order_items"]


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
5,3,keys,order_items,"dup_grain_natural:['order_id', 'product_id']",16,NaN,PASS,HARD_FAIL,"16 dòng trùng grain tự nhiên ['order_id', 'product_id'] — KHÔNG phải lỗi (dòng hàng hợp lệ, khác giá/số lượng); cần ...",0be18a18d00d41868b73904ffa6bf8f5
6,3,keys,order_items,surrogate_missing,1,0.0,FAIL,HARD_FAIL,Thiếu cột surrogate 'order_item_line_id' — chưa build line id ổn định (gọi ingestion.build_order_items_line_id trước).,0be18a18d00d41868b73904ffa6bf8f5


In [7]:
# (b) sau ingest_table -> có surrogate order_item_line_id -> keys() phải PASS cả 2 check
valid_oi, quarantine_oi, ingest_result_oi = ingest_table("order_items", batch_id=BATCH_ID)
print(ingest_result_oi)
r3b = q.keys(valid_oi, "order_items", BATCH_ID)
pd.DataFrame([r.__dict__ for r in r3b])


IngestResult(table='order_items', batch_id='0be18a18d00d41868b73904ffa6bf8f5', rows_in=714669, rows_valid=714669, rows_quarantined=0, ts='2026-08-06T02:42:54.805802+00:00', status='OK', quarantine_reasons={})


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,3,keys,order_items,"dup_grain_natural:['order_id', 'product_id']",16,NaN,PASS,HARD_FAIL,"16 dòng trùng grain tự nhiên ['order_id', 'product_id'] — KHÔNG phải lỗi (dòng hàng hợp lệ, khác giá/số lượng); cần ...",0be18a18d00d41868b73904ffa6bf8f5
1,3,keys,order_items,dup_surrogate:order_item_line_id,0,0.0,PASS,HARD_FAIL,0 dòng trùng surrogate key 'order_item_line_id'.,0be18a18d00d41868b73904ffa6bf8f5


## 4. Chiều 4 — `check_ri` (Referential integrity, QUARANTINE)

15 cặp FK con→cha (`schema.FK_CHECKS`, port nguyên phase0 PHẦN 4). Dòng mồ côi -> quarantine,
KHÔNG drop.


In [8]:
r4 = q.check_ri(tables, BATCH_ID)
all_results += r4
df4 = pd.DataFrame([r.__dict__ for r in r4])
df4


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,4,check_ri,order_items,ri_orphan:order_id->orders.order_id,0,0,PASS,QUARANTINE,"0/714,669 dòng (0.000%) có order_id không tồn tại trong orders.order_id.",0be18a18d00d41868b73904ffa6bf8f5
1,4,check_ri,order_items,ri_orphan:product_id->products.product_id,0,0,PASS,QUARANTINE,"0/714,669 dòng (0.000%) có product_id không tồn tại trong products.product_id.",0be18a18d00d41868b73904ffa6bf8f5
2,4,check_ri,orders,ri_orphan:customer_id->customers.customer_id,0,0,PASS,QUARANTINE,"0/646,945 dòng (0.000%) có customer_id không tồn tại trong customers.customer_id.",0be18a18d00d41868b73904ffa6bf8f5
3,4,check_ri,orders,ri_orphan:zip->geography.zip,0,0,PASS,QUARANTINE,"0/646,945 dòng (0.000%) có zip không tồn tại trong geography.zip.",0be18a18d00d41868b73904ffa6bf8f5
4,4,check_ri,customers,ri_orphan:zip->geography.zip,0,0,PASS,QUARANTINE,"0/121,930 dòng (0.000%) có zip không tồn tại trong geography.zip.",0be18a18d00d41868b73904ffa6bf8f5
5,4,check_ri,payments,ri_orphan:order_id->orders.order_id,0,0,PASS,QUARANTINE,"0/646,945 dòng (0.000%) có order_id không tồn tại trong orders.order_id.",0be18a18d00d41868b73904ffa6bf8f5
6,4,check_ri,shipments,ri_orphan:order_id->orders.order_id,0,0,PASS,QUARANTINE,"0/566,067 dòng (0.000%) có order_id không tồn tại trong orders.order_id.",0be18a18d00d41868b73904ffa6bf8f5
7,4,check_ri,returns,ri_orphan:order_id->orders.order_id,0,0,PASS,QUARANTINE,"0/39,939 dòng (0.000%) có order_id không tồn tại trong orders.order_id.",0be18a18d00d41868b73904ffa6bf8f5
8,4,check_ri,returns,ri_orphan:product_id->products.product_id,0,0,PASS,QUARANTINE,"0/39,939 dòng (0.000%) có product_id không tồn tại trong products.product_id.",0be18a18d00d41868b73904ffa6bf8f5
9,4,check_ri,reviews,ri_orphan:order_id->orders.order_id,0,0,PASS,QUARANTINE,"0/113,551 dòng (0.000%) có order_id không tồn tại trong orders.order_id.",0be18a18d00d41868b73904ffa6bf8f5


## 5. Chiều 5 — `completeness` (Completeness, WARN)

Null rate cột bắt buộc (`schema.required`, đã loại 2 ngoại lệ null hợp lệ:
`order_items.promo_id/promo_id_2`, `promotions.applicable_category`). Sai -> WARN, KHÔNG fail
cứng/quarantine.


In [9]:
r5 = []
for t, df in tables.items():
    r5 += q.completeness(df, t, BATCH_ID)
all_results += r5
df5 = pd.DataFrame([r.__dict__ for r in r5])
print("PASS/WARN:", df5["status"].value_counts().to_dict())
df5[df5["status"] != "PASS"]


PASS/WARN: {'PASS': 93}


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id


## 6. Chiều 6 — `business_rules` (Business-rule validity, QUARANTINE)

`revenue>=0`, `cogs>=0`, `quantity>0`, `order_date` trong biên [2012-07-04, 2022-12-31],
`order_status` ∈ enum, `discount_amount<=gross`. Port phase0 PHẦN 2.


In [10]:
r6 = q.business_rules(tables, BATCH_ID)
all_results += r6
pd.DataFrame([r.__dict__ for r in r6])


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,6,business_rules,sales,revenue_negative,0,0,PASS,QUARANTINE,0 dòng Revenue < 0 (rule: revenue>=0).,0be18a18d00d41868b73904ffa6bf8f5
1,6,business_rules,sales,cogs_negative,0,0,PASS,QUARANTINE,0 dòng COGS < 0 (rule: cogs>=0).,0be18a18d00d41868b73904ffa6bf8f5
2,6,business_rules,order_items,quantity_not_positive,0,0,PASS,QUARANTINE,0 dòng quantity <= 0 (rule: qty>0).,0be18a18d00d41868b73904ffa6bf8f5
3,6,business_rules,order_items,gross_lt_net,0,0,PASS,QUARANTINE,0 dòng discount_amount > gross (rule: gross>=net).,0be18a18d00d41868b73904ffa6bf8f5
4,6,business_rules,orders,status_enum_violation,0,0,PASS,QUARANTINE,0 dòng order_status ngoài enum (rule: status ∈ enum).,0be18a18d00d41868b73904ffa6bf8f5
5,6,business_rules,orders,order_date_out_of_bounds,0,0,PASS,QUARANTINE,"0 dòng order_date ngoài [2012-07-04, 2022-12-31] (rule: date trong biên).",0be18a18d00d41868b73904ffa6bf8f5


## 7. Chiều 7 — `time_coverage` (Time coverage, WARN)

Phủ ngày liên tục trong biên kỳ vọng, đếm ngày thiếu — port phase0 PHẦN 3. Cùng mapping
`table -> date_col` dùng trong `quality.run_all()`.


In [11]:
date_cols_by_table = {
    "orders": "order_date", "sales": "Date", "web_traffic": "date",
    "shipments": "ship_date", "returns": "return_date", "reviews": "review_date",
    "inventory": "snapshot_date", "customers": "signup_date",
}
r7 = []
for t, col in date_cols_by_table.items():
    if t in tables and col in tables[t].columns:
        r7 += q.time_coverage(tables[t], t, col, BATCH_ID)
all_results += r7
pd.DataFrame([r.__dict__ for r in r7])


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,7,time_coverage,orders,missing_days:order_date,0,0,PASS,WARN,"0 ngày thiếu trong [2012-07-04, 2022-12-31].",0be18a18d00d41868b73904ffa6bf8f5
1,7,time_coverage,sales,missing_days:Date,0,0,PASS,WARN,"0 ngày thiếu trong [2012-07-04, 2022-12-31].",0be18a18d00d41868b73904ffa6bf8f5
2,7,time_coverage,web_traffic,missing_days:date,0,0,PASS,WARN,"0 ngày thiếu trong [2013-01-01, 2022-12-31].",0be18a18d00d41868b73904ffa6bf8f5
3,7,time_coverage,shipments,missing_days:ship_date,0,0,PASS,WARN,"0 ngày thiếu trong [2012-07-04, 2022-12-29].",0be18a18d00d41868b73904ffa6bf8f5
4,7,time_coverage,returns,missing_days:return_date,20,0,WARN,WARN,"20 ngày thiếu trong [2012-07-11, 2022-12-31] (vd: 2012-07-13, 2019-03-09, 2020-02-04, 2020-02-08, 2020-02-23...).",0be18a18d00d41868b73904ffa6bf8f5
5,7,time_coverage,reviews,missing_days:review_date,2,0,WARN,WARN,"2 ngày thiếu trong [2012-07-10, 2022-12-31] (vd: 2012-07-11, 2021-02-22).",0be18a18d00d41868b73904ffa6bf8f5
6,7,time_coverage,inventory,missing_days:snapshot_date,3680,0,WARN,WARN,"3680 ngày thiếu trong [2012-07-31, 2022-12-31] (vd: 2012-08-01, 2012-08-02, 2012-08-03, 2012-08-04, 2012-08-05...).",0be18a18d00d41868b73904ffa6bf8f5
7,7,time_coverage,customers,missing_days:signup_date,5,0,WARN,WARN,"5 ngày thiếu trong [2012-07-04, 2022-12-31] (vd: 2012-07-06, 2012-07-13, 2012-07-15, 2012-08-20, 2012-09-28).",0be18a18d00d41868b73904ffa6bf8f5


## 8. Chiều 8 — `dup_outlier` (Duplicate & outlier, QUARANTINE / FLAG)

Trùng FULL-ROW trên business columns -> QUARANTINE. Outlier IQR (k=3) trên cột numeric -> chỉ
FLAG (WARN), không quarantine.


In [12]:
r8 = []
for t, df in tables.items():
    r8 += q.dup_outlier(df, t, BATCH_ID)
all_results += r8
df8 = pd.DataFrame([r.__dict__ for r in r8])
print("PASS/WARN/FAIL:", df8["status"].value_counts().to_dict())
df8[df8["status"] != "PASS"]


PASS/WARN/FAIL: {'WARN': 17, 'PASS': 14}


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
3,8,dup_outlier,products,outlier_iqr:price,2,NaN,WARN,QUARANTINE,"2 dòng ngoài [-22923.76, 30703.72] (IQR x3.0) ở cột 'price' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
4,8,dup_outlier,products,outlier_iqr:cogs,3,NaN,WARN,QUARANTINE,"3 dòng ngoài [-17454.48, 23354.47] (IQR x3.0) ở cột 'cogs' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
6,8,dup_outlier,promotions,outlier_iqr:discount_value,5,NaN,WARN,QUARANTINE,"5 dòng ngoài [-12.00, 44.00] (IQR x3.0) ở cột 'discount_value' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
9,8,dup_outlier,order_items,outlier_iqr:unit_price,177,NaN,WARN,QUARANTINE,"177 dòng ngoài [-14193.72, 23374.37] (IQR x3.0) ở cột 'unit_price' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
10,8,dup_outlier,order_items,outlier_iqr:discount_amount,66211,NaN,WARN,QUARANTINE,"66211 dòng ngoài [-2902.89, 3870.52] (IQR x3.0) ở cột 'discount_amount' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
12,8,dup_outlier,payments,outlier_iqr:payment_value,2843,NaN,WARN,QUARANTINE,"2843 dòng ngoài [-70394.81, 111782.22] (IQR x3.0) ở cột 'payment_value' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
14,8,dup_outlier,shipments,outlier_iqr:shipping_fee,76050,NaN,WARN,QUARANTINE,"76050 dòng ngoài [-4.32, 7.79] (IQR x3.0) ở cột 'shipping_fee' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
16,8,dup_outlier,returns,outlier_iqr:refund_amount,820,NaN,WARN,QUARANTINE,"820 dòng ngoài [-36352.39, 56807.77] (IQR x3.0) ở cột 'refund_amount' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
19,8,dup_outlier,inventory,outlier_iqr:stock_on_hand,3253,NaN,WARN,QUARANTINE,"3253 dòng ngoài [-570.00, 795.00] (IQR x3.0) ở cột 'stock_on_hand' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5
20,8,dup_outlier,inventory,outlier_iqr:units_received,3321,NaN,WARN,QUARANTINE,"3321 dòng ngoài [-49.00, 70.00] (IQR x3.0) ở cột 'units_received' — FLAG, không quarantine.",0be18a18d00d41868b73904ffa6bf8f5


## 9. Chiều 9 — `reconcile` (Cross-table reconciliation, HARD_FAIL)

Check CHÍNH THỨC (target definition, ngưỡng MAPE 1.0%, port nguyên phase0 PHẦN 5):
`sales.csv` (GROSS, dùng đối chiếu forecast Phần B) vs `order_items` tái dựng theo 3 định nghĩa
mask (chọn khớp nhất). `payments` vs `sales` chỉ diagnostic bổ sung (KHÔNG gate).


In [13]:
r9 = q.reconcile(tables, BATCH_ID)
all_results += r9
pd.DataFrame([r.__dict__ for r in r9])


,dimension,name,table,metric,value,threshold,status,severity,detail,batch_id
0,9,reconcile,sales<->order_items,revenue_mape:gross_all_orders,0.0000,1.0,PASS,HARD_FAIL,Định nghĩa khớp nhất 'gross_all_orders': MAPE ngày Revenue 0.000%.,0be18a18d00d41868b73904ffa6bf8f5
1,9,reconcile,sales<->order_items,cogs_mape:gross_all_orders,0.0000,1.0,PASS,HARD_FAIL,COGS tái dựng (mask 'gross_all_orders'): MAPE ngày 0.000%.,0be18a18d00d41868b73904ffa6bf8f5
2,9,reconcile,sales<->payments,revenue_mape,5.1685,10.0,PASS,HARD_FAIL,"payments.payment_value theo ngày vs sales.Revenue: MAPE 5.168% (diagnostic — payment date khác order_date, KHÔNG dùn...",0be18a18d00d41868b73904ffa6bf8f5


## 10. Chiều 10 — `write_dq_log` (Data-quality log, INFO)

Gộp cả 9 chiều trên thành 1 log versioned (`docs/dq_log/dq_log_<batch_id>.csv` +
`dq_log_latest.csv` — thay `ops.dq_log` Postgres thật tới khi M2 dựng infra, CÙNG schema cột).


In [14]:
log_df = q.write_dq_log(all_results, BATCH_ID)
print(f"{len(log_df)} dòng log, batch_id={BATCH_ID}")
print("Ghi vào:", q.DQ_LOG_DIR / f"dq_log_{BATCH_ID}.csv")
log_df["status"].value_counts()


199 dòng log, batch_id=0be18a18d00d41868b73904ffa6bf8f5
Ghi vào: /Users/lhoanghai_/Documents/Study/Dự án datathon2026/docs/dq_log/dq_log_0be18a18d00d41868b73904ffa6bf8f5.csv


status
PASS    177
WARN     21
FAIL      1
Name: count, dtype: int64

In [15]:
gate = q.gate_status(all_results)
print("GATE STATUS (10 chiều, gọi rời từng hàm ở notebook này):", gate)
log_df.groupby(["dimension", "dimension_name", "status"]).size().unstack(fill_value=0)


GATE STATUS (10 chiều, gọi rời từng hàm ở notebook này): FAIL


,status,FAIL,PASS,WARN
dimension,dimension_name,,,
1,inventory,0,14,0
2,validate_schema,0,14,0
3,keys,1,14,0
4,check_ri,0,15,0
5,completeness,0,93,0
6,business_rules,0,6,0
7,time_coverage,0,4,4
8,dup_outlier,0,14,17
9,reconcile,0,3,0


## 11. Đối chiếu — `quality.run_all()` (chạy gộp, đúng hàm production dùng ở M2/M5)

Chạy lại bằng đúng entrypoint mà job ingest (M2) / test (M5) sẽ gọi, để xác nhận kết quả khớp
với việc gọi rời từng hàm ở trên (khác biệt kỳ vọng: `run_all()` load raw thô — `order_items`
CHƯA có surrogate -> chiều 3 FAIL đúng thiết kế, giống mục 3(a) ở trên).


In [16]:
full_log, full_gate = q.run_all()
print("run_all() gate:", full_gate)
print(f"run_all() log rows: {len(full_log)}")
full_log["status"].value_counts()


run_all() gate: FAIL
run_all() log rows: 199


status
PASS    177
WARN     21
FAIL      1
Name: count, dtype: int64

In [17]:
full_log[full_log["status"] == "FAIL"][["dimension", "dimension_name", "table", "metric", "value", "detail"]]


,dimension,dimension_name,table,metric,value,detail
26,3,keys,order_items,surrogate_missing,1.0,Thiếu cột surrogate 'order_item_line_id' — chưa build line id ổn định (gọi ingestion.build_order_items_line_id trước).


## Kết luận

- 10/10 hàm DQ trong `datathon.quality` chạy được trên toàn bộ 14 bảng raw thật (không mock).
- FAIL duy nhất mang tính THIẾT KẾ: `order_items` thiếu surrogate `order_item_line_id` khi đọc
  raw thô trực tiếp (chiều 3) — biến mất sau khi qua `ingestion.ingest_table()` (mục 3b), đúng
  pipeline thật (M2 luôn ingest trước khi MERGE Iceberg).
- `sales <-> order_items` reconcile (chiều 9, chỉ tiêu chính thức) MAPE Revenue/COGS theo ngày —
  xem số thật ở mục 9 (kỳ vọng khớp `.process_status/M4a.md`: 0.000% với định nghĩa
  `gross_all_orders`).
- Log đầy đủ đã ghi `docs/dq_log/dq_log_<batch_id>.csv` (runtime output, không phải file nguồn —
  không commit cùng notebook).

**Hạn chế**: DQ ở đây chạy tầng pandas trên CSV raw (không phải Iceberg/Trino thật) — dùng đúng
logic sẽ chạy trong job M2, nhưng số liệu partition/snapshot Iceberg (nếu có) phải verify riêng
qua `jobs/` + Trino (đã làm ở M2 live-verify, xem `PROCESS.md`).
